[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/internshipinabook/mlops-internship-in-a-book/blob/main/notebooks/week3_root_cause_analysis_STARTER.ipynb)


[![Get the Book on Selar](https://img.shields.io/badge/Get%20the%20full%20book-Selar-1E7A34?style=for-the-badge)](https://selar.com/7m27877571)

# Week 3 -- Root Cause Analysis (STARTER)
### The MLOps Internship | DataFlow Infrastructure

**Dataset:** CreditDefault-NG (UCI Credit Card Default)

**This week:** SHAP feature importance over time, slice analysis, upstream ETL audit, root cause brief

STARTER -- complete each exercise before moving on.


In [ ]:
# -- Colab Setup (skip if running locally) -------------------------
import os
if not os.path.exists('requirements.txt'):
    !git clone https://github.com/internshipinabook/mlops-internship-in-a-book.git mlops-book
    %cd mlops-book
    !pip install -r requirements.txt -q
os.makedirs('outputs', exist_ok=True)
os.makedirs('models',  exist_ok=True)
os.makedirs('datasets', exist_ok=True)
print('Setup complete.')


In [ ]:
# -- Reproducibility Seeds ------------------------------------------
# SEED=42 ensures every random operation gives the same result on every run.
# Set seeds BEFORE any data loading, model creation, or training.
import random, numpy as np
SEED = 42
random.seed(SEED)       # Python random
np.random.seed(SEED)    # NumPy random (used by sklearn)
print(f'Seeds fixed: {SEED}')


## Learning Objectives

- Compute SHAP values for the Q1 model on Q1 data and Q4 data separately
- Identify which features changed their SHAP contribution between Q1 and Q4
- Perform slice analysis: compute model AUC for LIMIT_BAL quartile segments
- Audit the upstream ETL pipeline to identify where the LIMIT_BAL shift originated
- Write the root cause brief: one root cause, three evidence points, no hypothesis language



---

## MONDAY -- "SHAP Feature Importance Over Time"


Running the Q1 model on Q4 data and computing SHAP shows which features are driving Q4 predictions. Comparing Q1 SHAP (model on Q1 data) with Q4 SHAP (model on Q4 data) reveals feature importance drift. The model weights are unchanged -- but because the input distribution shifted, different features now dominate the predictions.

Pause and Predict -- before computing: if LIMIT_BAL drifted the most (highest PSI from Week 2), do you expect it also has the largest SHAP importance increase? Or could a feature with moderate PSI have a larger SHAP shift? Explain your reasoning.


### Exercise 3.1 -- SHAP comparison table

Compute mean |SHAP| per feature for Q1 and Q4. Build a table with delta and % change. Which three features changed most? Do they overlap with the top PSI features from Week 2?


In [ ]:
# Exercise 3.1: SHAP comparison table
# ----------------------------------------------------------
# Hint: read the markdown cell above carefully.
# Hint: check the Common Mistakes section at the end.

# YOUR CODE HERE


**Monday scaffold** (read and run, then adapt in exercises):


In [ ]:
# -- Monday: SHAP Feature Importance Over Time --
import shap, mlflow.sklearn, pandas as pd, numpy as np

model = mlflow.sklearn.load_model('models:/credit-default-v2/Production')
q1 = pd.read_csv('datasets/creditdefault_q1.csv')
q4 = pd.read_csv('datasets/creditdefault_q4.csv')
feat_cols = [c for c in q1.columns if c != 'DEFAULT_NEXT_MONTH']

explainer = shap.TreeExplainer(model)
shap_q1   = explainer.shap_values(q1[feat_cols])  # (n_q1, n_features)
shap_q4   = explainer.shap_values(q4[feat_cols])  # (n_q4, n_features)

df = pd.DataFrame({'Q1': np.abs(shap_q1).mean(axis=0),
                   'Q4': np.abs(shap_q4).mean(axis=0)}, index=feat_cols)
df['delta']      = df.Q4 - df.Q1
df['pct_change'] = (df.delta / df.Q1 * 100).round(1)
print(df.sort_values('delta', ascending=False).to_string())


### Expected Output (illustrative)

```
              Q1      Q4    delta  pct_change
LIMIT_BAL  0.0412  0.1187  0.0775      188.1
PAY_0      0.0389  0.0512  0.0123       31.6
BILL_AMT1  0.0201  0.0248  0.0047       23.4
...
```
`LIMIT_BAL`'s mean |SHAP value| nearly tripled between Q1 and Q4 — the model is now leaning on
this feature far more heavily than it did at launch, which is consistent with the PSI finding
from Week 2 rather than contradicting it.


### Monday Morning Moment

*Slack -- Monday, 11:30am.*

**Temi Adeyemi:** SHAP comparison done. LIMIT_BAL importance up 41% from Q1 to Q4.

**Dr. Emeka Obi:** What does a 41% increase in SHAP importance mean?

**Temi Adeyemi:** The model is leaning more heavily on credit limit in Q4. High-limit customers get higher default probability -- but the model was not trained on enough of them to make that judgment accurately.

**Sola Fashola:** Not just the distribution shifted -- the model's reliance on it shifted too.

**Temi Adeyemi:** Both changed simultaneously. That is what makes this a root cause, not just a drift symptom.

**Dr. Emeka Obi:** Find the upstream origin. Where are these high-limit customers coming from?




---

## TUESDAY -- "Slice Analysis"


Compute model AUC separately for LIMIT_BAL below NT$100K, NT$100K-400K, and above NT$400K. If the model performs well on standard-limit customers but poorly on high-limit customers, that slice failure directly corresponds to the PSI finding from Week 2. The slice gap quantifies the business impact of the drift.


### Exercise 3.2 -- Slice analysis matrix

Compute AUC for: (a) LIMIT_BAL quartiles, (b) AGE quartiles, (c) PAY_0 status groups (-1, 0, 1, 2). Which slice has lowest AUC? Which has the largest Q1-to-Q4 AUC gap?


In [ ]:
# Exercise 3.2: Slice analysis matrix
# ----------------------------------------------------------
# Hint: read the markdown cell above carefully.
# Hint: check the Common Mistakes section at the end.

# YOUR CODE HERE


**Tuesday scaffold** (read and run, then adapt in exercises):


In [ ]:
# -- Tuesday: Slice Analysis --
from sklearn.metrics import roc_auc_score
import pandas as pd

q4    = pd.read_csv('datasets/creditdefault_q4.csv')
X     = q4.drop('DEFAULT_NEXT_MONTH', axis=1)
y     = q4['DEFAULT_NEXT_MONTH']
y_pred = model.predict_proba(X)[:, 1]

slices = {
    'standard  (<100K)':  q4.LIMIT_BAL < 100_000,
    'mid-range (100K-400K)': (q4.LIMIT_BAL >= 100_000) & (q4.LIMIT_BAL < 400_000),
    'high-limit (>400K)':  q4.LIMIT_BAL >= 400_000,
}
print('Slice Analysis -- AUC by credit limit segment:')
for name, mask in slices.items():
    n = mask.sum()
    if n < 20: continue
    auc = roc_auc_score(y[mask], y_pred[mask])
    print(f'  {name}: AUC={auc:.4f} (n={n})')


### Expected Output (illustrative)

```
Slice Analysis -- AUC by credit limit segment:
  standard  (<100K): AUC=0.9118 (n=2144)
  mid-range (100K-400K): AUC=0.8734 (n=1867)
  high-limit (>400K): AUC=0.8034 (n=489)
```
The high-limit slice's AUC (0.8034) sits well below the standard-limit slice (0.9118) — this
is the concrete evidence that the drift found in Week 2 is *hurting predictive performance*,
not just changing the input distribution harmlessly.



---

## WEDNESDAY -- "Upstream ETL Audit"


The root cause of drift is almost always upstream of the model. The CreditDefault-NG data comes from a credit bureau pipeline. When was the ETL last changed? Is LIMIT_BAL computed the same way in Q4 as in Q1? 'The data drifted' is a symptom. 'The bureau began including a premium account segment in August' is a root cause.


### Exercise 3.3 -- ETL audit

Read etl/transform.py. Find every place LIMIT_BAL is modified. Use git blame to date each change. Does any Q2-Q3 change explain the distribution shift?


In [ ]:
# Exercise 3.3: ETL audit
# ----------------------------------------------------------
# Hint: read the markdown cell above carefully.
# Hint: check the Common Mistakes section at the end.

# YOUR CODE HERE


**Wednesday scaffold** (read and run, then adapt in exercises):


In [ ]:
# -- Wednesday: Upstream ETL Audit --
import subprocess

# Check when LIMIT_BAL handling changed in the ETL
r = subprocess.run(['grep', '-n', 'LIMIT_BAL', 'etl/transform.py'],
                   capture_output=True, text=True)
print('LIMIT_BAL in ETL:')
print(r.stdout)

# Check git history
r2 = subprocess.run(['git', 'log', '--oneline', 'etl/transform.py'],
                    capture_output=True, text=True)
print('\nETL history:')
print(r2.stdout[:800])


### Expected Output (illustrative)

```
LIMIT_BAL in ETL:
transform.py:47:    df['LIMIT_BAL'] = df['LIMIT_BAL'].clip(upper=CREDIT_CAP)

ETL history:
a91f3c2 Raise CREDIT_CAP to accommodate new premium tier (2026-03-14)
7d4e881 Initial LIMIT_BAL clipping logic (2025-11-02)
```
The commit on 2026-03-14 — after the Q1 training data was collected — is the upstream origin:
a new premium credit tier was introduced into the pipeline without the model being retrained
to see it.



---

## THURSDAY -- "The Root Cause Statement"


One sentence. No hedges. '[Event] caused [distribution shift] which caused [model behaviour change] which caused [business impact].' Then three evidence points. Then the counterfactual validation: remove the new customer segment from Q4 and verify the drift disappears.

Pause and Predict -- write your root cause statement now, before computing the counterfactual. Then check whether the counterfactual confirms or contradicts it. If it contradicts, what would you change?


### Exercise 3.4 -- Root cause validation

Remove high-limit accounts from Q4. Re-run PSI for LIMIT_BAL. Does it drop below 0.10? If not, what additional filtering is needed?


In [ ]:
# Exercise 3.4: Root cause validation
# ----------------------------------------------------------
# Hint: read the markdown cell above carefully.
# Hint: check the Common Mistakes section at the end.

# YOUR CODE HERE


**Thursday scaffold** (read and run, then adapt in exercises):


In [ ]:
# -- Thursday: The Root Cause Statement --
# Root cause validation
import pandas as pd

q1 = pd.read_csv('datasets/creditdefault_q1.csv')
q4 = pd.read_csv('datasets/creditdefault_q4.csv')

# Remove the new high-limit segment from Q4
q4_filtered = q4[q4.LIMIT_BAL < 400_000]

psi_original = compute_psi(q1.LIMIT_BAL.dropna(), q4.LIMIT_BAL.dropna())
psi_filtered  = compute_psi(q1.LIMIT_BAL.dropna(), q4_filtered.LIMIT_BAL.dropna())

print(f'PSI with full Q4:          {psi_original:.4f}')
print(f'PSI without high-limit:    {psi_filtered:.4f}')
print(f'Root cause validated: {psi_filtered < 0.10}')


### Expected Output (illustrative)

```
PSI with full Q4:          0.3812
PSI without high-limit:    0.0891
Root cause validated: True
```
Removing the high-limit segment alone drops `LIMIT_BAL` PSI from severe (0.38) to stable
(0.09) — this is the validation step that turns "we suspect the high-limit segment" into "we
have confirmed the high-limit segment," per this week's root-cause discipline.



---

## FRIDAY -- "The Friday Build: Root Cause Brief"


One page. Root cause statement, three evidence points with numbers, upstream origin, timeline, recommended fix. No hypothesis language. Write it as if you are presenting to an engineering team that has ten minutes and will act on what you write.


**Friday scaffold** (read and run, then adapt in exercises):


In [ ]:
# -- Friday: The Friday Build: Root Cause Brief --
# Root cause brief structure:
# ROOT CAUSE: [one sentence, no hedges]
# EVIDENCE 1: LIMIT_BAL PSI = X.XX (severe drift, Q1 vs Q4)
# EVIDENCE 2: LIMIT_BAL SHAP importance changed by X% (Q1 vs Q4)
# EVIDENCE 3: model AUC = X.XX on high-limit vs X.XX on standard-limit
# UPSTREAM ORIGIN: [which ETL change or bureau update, with date]
# TIMELINE: drift onset [quarter], current severity [severe/moderate]
# RECOMMENDED FIX: retrain with Q2-Q4 data including high-limit segment
# RISK IF UNRESOLVED: ~1,840 Q4 decisions at elevated unreliability risk


### Expected Output (illustrative)

No code output — Friday's deliverable is the written root-cause brief. It should read as a
single unhedged sentence backed by the three evidence points computed Monday-Thursday, e.g.:
*"Root cause: introduction of a premium credit-limit tier in March 2026 shifted the LIMIT_BAL
distribution (PSI 0.38), which the production model — trained before the tier existed — was
never retrained to handle."*


### Friday Workplace Moment

*DataFlow Infrastructure -- Friday standup.*

**Dr. Emeka Obi:** Root cause. One sentence.

**Temi Adeyemi:** The credit bureau began including a premium account segment in August, adding high-limit customers above NT$400,000 to the data pipeline that the Q1 model was not trained on, causing it to systematically misestimate default risk for this segment.

**Dr. Emeka Obi:** Three evidence points?

**Temi Adeyemi:** LIMIT_BAL PSI 0.38 -- severe drift. SHAP importance up 41%. Slice AUC 0.67 on high-limit vs 0.88 on standard-limit customers.

**Sola Fashola:** Counterfactual?

**Temi Adeyemi:** PSI drops from 0.38 to 0.08 when high-limit customers are removed from Q4. Validates the root cause.

**Dr. Emeka Obi:** That is a root cause brief. Merge it.



## YOUR WEEK 3 CHECKLIST

Check each item before moving on.

- [ ] I can compute SHAP feature importance for any sklearn model and compare it across two datasets.
- [ ] I ran slice analysis on at least three dimensions and can state the worst-performing slice with its AUC.
- [ ] I identified the upstream ETL change that caused the distribution shift.
- [ ] My root cause statement is one sentence with no hedges, supported by three evidence points.
- [ ] I validated the root cause by testing the counterfactual.


## Challenge Task

> **Core path:** Implement a SHAP monitoring pipeline: compute mean |SHAP| per feature for each weekly Q2-Q4 cohort. Plot LIMIT_BAL importance over time. At which week does it first exceed the Q1 baseline by 20%?

> **Advanced path:** Compute Shapley Data Values for Q1 training examples. Which examples have the most negative Shapley values? Are they disproportionately standard-limit accounts that now confuse the model?


## Common Mistakes

**Conflating feature importance with feature drift:** High PSI does not mean high SHAP importance. A feature can have PSI=0.38 but low SHAP -- the model barely uses it. Both matter, separately.

**Slice analysis without confidence intervals:** AUC 0.67 on 80 high-limit customers vs 0.88 on 1,200 standard-limit customers needs bootstrap CIs before claiming the gap is significant.

**Root cause with multiple causes:** 'LIMIT_BAL drift AND ETL changes AND model limitations' is not a root cause. Find the single upstream event.



---

**The MLOps Internship** -- Book 4 of 9, InternshipInABook™

Dataset: CreditDefault-NG | Company: DataFlow Infrastructure, Lagos/Abuja

[internshipinabook.com](https://internshipinabook.com)


📘 **Get the complete illustrated book (all 13 weeks, DOCX + EPUB):** [https://selar.com/7m27877571](https://selar.com/7m27877571)